In [ ]:
# Importar las librerías necesarias
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from xgboost import XGBClassifier


In [ ]:
# Cargar las matrices de características de entrenamiento y prueba
X_train = pd.read_parquet('../data/X_train.parquet', engine='fastparquet')
X_test = pd.read_parquet('../data/X_test.parquet', engine='fastparquet')

y_train = pd.read_parquet('../data/y_train.parquet', engine='fastparquet')['is_fraud']
y_test = pd.read_parquet('../data/y_test.parquet', engine='fastparquet')['is_fraud']
    
# Verificar las dimensiones de los conjuntos de datos
print(f"X_train shape: {X_train.shape} | X_test shape: {X_test.shape}")

In [ ]:
# Escalado de características: la Regresión Logística es sensible a las diferencias de escala
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Inicializar Regresión Logística usando 'class_weight=balanced' para tratar el desbalance de clases
logic_model = LogisticRegression(class_weight='balanced', max_iter=1000)

# Entrenar el modelo base (baseline)
logic_model.fit(X_train_scaled, y_train)

In [ ]:
# Generar predicciones en el conjunto de prueba
y_pred = logic_model.predict(X_test_scaled)

# Mostrar la matriz de confusión y el reporte de clasificación
print("=== CONFUSION MATRIX (LOGISTIC REGRESSION) ===")
print(confusion_matrix(y_test, y_pred))
print("\n=== CLASSIFICATION REPORT (LOGISTIC REGRESSION) ===")
print(classification_report(y_test, y_pred))

**Evaluación de la Regresión Logística**: La precisión y el F1-Score para la clase positiva (fraude) resultaron insuficientes para un entorno de producción. Se cambia la estrategia a XGBoost para capturar fronteras de decisión no lineales.

In [ ]:
# Calcular el factor de peso de clases para compensar el desbalance (Casos Negativos / Casos Positivos)
class_counts = y_train.value_counts()
scale_weight = class_counts[0] / class_counts[1]

# Inicializar XGBoost con configuración para balanceo de clases
xgb_model = XGBClassifier(
    n_estimators=300, 
    max_depth=5, 
    scale_pos_weight=scale_weight,
    random_state=42,
    n_jobs=-1
)

# Entrenar XGBoost utilizando las características originales (no requiere escalado de datos)
xgb_model.fit(X_train, y_train)

In [ ]:
# Generar predicciones estándar usando el umbral por defecto (0.50)
y_pred_xgb = xgb_model.predict(X_test)

print("=== CONFUSION MATRIX (XGBOOST - THRESHOLD 0.50) ===")
print(confusion_matrix(y_test, y_pred_xgb))
print("\n=== CLASSIFICATION REPORT (XGBOOST - THRESHOLD 0.50) ===")
print(classification_report(y_test, y_pred_xgb))

In [ ]:
# Evaluar predicciones en el conjunto de entrenamiento para medir posible sobreajuste (overfitting)
y_pred_train = xgb_model.predict(X_train)

print("=== TRAINING SET REPORT (OVERFITTING EVALUATION) ===")
print(classification_report(y_train, y_pred_train))

In [ ]:
# Obtener probabilidades continuas de fraude (clase 1) en lugar de predicciones binarias fijas
y_probs = xgb_model.predict_proba(X_test)[:, 1]

# Evaluar usando un umbral de 0.60 para reducir falsos positivos
y_pred_60 = (y_probs >= 0.60).astype(int)

print("=== THRESHOLD 0.60 ===")
print(confusion_matrix(y_test, y_pred_60))
print(classification_report(y_test, y_pred_60))

In [ ]:
# Evaluar usando un umbral de 0.70 para optimizar la precisión manteniendo un alto recall
y_pred_70 = (y_probs >= 0.70).astype(int)

print("=== THRESHOLD 0.70 ===")
print(confusion_matrix(y_test, y_pred_70))
print(classification_report(y_test, y_pred_70))

In [ ]:
# El mejor rendimiento se obtuvo con el umbral de 0.70.
# La validación cruzada conservará este umbral al calcular el F1-score.
# También evaluamos el modelo independientemente del umbral mediante el promedio de precisión (PR-AUC).

# Almacenar las métricas de rendimiento para cada uno de los 5 pliegues (folds) de validación
pr_auc_scores = []
f1_scores_70 = [] 

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for train_idx, val_idx in skf.split(X_train, y_train):

    # Separar datos de entrenamiento y validación para el pliegue actual
    X_tr, X_val = X_train.iloc[train_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    # Entrenar XGBoost en el pliegue actual
    xgb_model.fit(X_tr, y_tr)

    # Obtener probabilidades de fraude en el conjunto de validación
    probs = xgb_model.predict_proba(X_val)[:, 1]

    # Calcular PR-AUC (independiente del umbral de clasificación)
    pr_auc_scores.append(average_precision_score(y_val, probs))

    # Convertir probabilidades en predicciones binarias usando el umbral optimizado de 0.70
    preds_70 = (probs >= 0.70).astype(int)

    # Calcular F1-score en el pliegue de validación
    score = f1_score(y_val, preds_70)
    f1_scores_70.append(score)

# Mostrar la media y desviación estándar de las métricas en la validación cruzada
print("=== FINAL CROSS-VALIDATION RESULTS (5-FOLD) ===")
print("Mean PR-AUC:", np.mean(pr_auc_scores), "±", np.std(pr_auc_scores))
print("Mean F1-Score (Threshold = 0.70):", np.mean(f1_scores_70), "±", np.std(f1_scores_70))

In [ ]:
# Crear una copia de X_test para evitar modificar el dataframe original
df_dashboard = X_test.copy()

# Agregar la etiqueta real
df_dashboard['is_fraud_real'] = y_test.values

# Agregar las probabilidades continuas de fraude predichas por XGBoost
df_dashboard['fraud_probability'] = y_probs

# Agregar las predicciones binarias usando el umbral optimizado de 0.70
df_dashboard['model_prediction'] = y_pred_70

# Agregar un indicador de error para simplificar el rastreo de clasificaciones incorrectas (1 = Error, 0 = Correcto)
df_dashboard['prediction_error'] = (df_dashboard['is_fraud_real'] != df_dashboard['model_prediction']).astype(int)

# Exportar el dataframe consolidado para el tablero de Power BI
df_dashboard.to_csv('../data/fraud_results_powerbi.csv', index=False)

print("File 'fraud_results_powerbi.csv' generated successfully!")
print(f"Total records exported for dashboarding: {len(df_dashboard)}")

### Justificación del Modelo y Estrategia de Decisión

1. **Modelo Base con Regresión Logística**:
   Se utilizó como punto de comparación (benchmark). Aunque la configuración `class_weight='balanced'` mejoró la detección de fraudes, el modelo generó un número inaceptable de falsos positivos (baja precisión), lo que en un entorno de producción bloquearía transacciones legítimas innecesariamente.

2. **Ventajas de XGBoost y Gradient Boosting:**  
   XGBoost captura relaciones complejas y no lineales entre las variables, como combinaciones entre la hora de la transacción, la distancia geográfica y el monto. El parámetro `scale_pos_weight` atiende directamente el desbalance severo de clases.

3. **Optimización del Umbral de Decisión:**  
   Los clasificadores estándar usan un umbral de probabilidad por defecto de 0.50. Incrementar este umbral a **0.70** reduce las falsas alarmas manteniendo una alta capacidad de detección (recall) de transacciones fraudulentas.

4. **Validación Cruzada Estratificada (Stratified K-Fold):**  
   Debido al desbalance extremo de clases (~0.52% de casos positivos), una validación cruzada tradicional generaría distribuciones inconsistentes de fraude entre pliegues. `StratifiedKFold` con 5 particiones garantiza una evaluación confiable mediante métricas **PR-AUC** y **F1-Score**.